As starting point we take the SpatialData `.zarr` file created with Harpy, and preprocess it, so it can be visualized by Vitessce.

In [1]:
import json
import os
import numpy as np
from spatialdata import read_zarr
from dataclasses import dataclass

from vitessce import (
    VitessceConfig,
    Component as cm,
    AnnDataWrapper,
    MultiImageWrapper,
    OmeZarrWrapper
)
from vitessce.data_utils import (
    VAR_CHUNK_SIZE,
    multiplex_img_to_ome_zarr,
    optimize_adata,
)

c:\git\2024-07-UTBenchmark-SpC\vitessce\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\git\2024-07-UTBenchmark-SpC\vitessce\.venv\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
@dataclass
class Dataset:
    name: str
    description: str

In [ ]:
def make_vitessce_dataset(name: str,
                          description: str,
                          spatialdata_filepath: str,
                          spaceranger_output_path: str,
                          bin_size_um: int,
                          output_dir: str) -> None:
    
    output_path_adata = os.path.join(output_dir, "vitessce", "A", "1", "adata_{bin_size:03}um.zarr")
    output_path_img = os.path.join(output_dir, "vitessce", "A", "1", "full_image.ome.zarr")
    output_path_tissue_mask = os.path.join(output_dir, "vitessce", "A", "1", "tissue_mask.ome.zarr")
    output_path_vitessce_config = os.path.join(output_dir, f"vitessce_config_{name}.json")

    sdata = read_zarr(spatialdata_filepath)

    bin_side_length = _get_visium_hd_spot_diameter(spaceranger_output_path)
    
    _write_zarrs_for_vitessce(sdata,
                              name,
                              bin_size_um,
                              bin_side_length,
                              output_path_adata,
                              output_path_img,
                              output_path_tissue_mask)
    
    vc = _make_vitessce_config(name,
                               description,
                               output_path_adata,
                               output_path_img,
                               output_path_tissue_mask)

    with open(output_path_vitessce_config, "w") as f:
        json.dump(vc.to_dict(), f, indent=4)
    

def _to_square(x, y, side):
    """
    Convert an (x, y) coordinate to a polygon (square) with a given side length.
    """
    r = side / 2
    return np.array([
        [x - r, y + r],
        [x + r, y + r],
        [x + r, y - r],
        [x - r, y - r]
    ])


def _get_visium_hd_spot_diameter(spaceranger_output_path: str,
                                 bin_size_um: int) -> float:
    scalefactors_filepath = os.path.join(spaceranger_output_path, f"outs/binned_outputs/square_{bin_size_um:03}um/spatial/scalefactors_json.json")
    
    # Radius calculation is identical to https://github.com/scverse/spatialdata-io/blob/main/src/spatialdata_io/readers/visium_hd.py#L318
    with open(scalefactors_filepath) as file:
        scalefactors_bins = json.load(file)

    diameter = scalefactors_bins["spot_diameter_fullres"]  # CHECKME: in what units?   
    return diameter 


def _write_zarrs_for_vitessce(sdata,
                              dataset_id: str,
                              bin_size_um: int,
                              bin_side_length: float,
                              output_path_adata: str,
                              output_path_img: str,
                              output_path_tissue_mask: str):

    # Create AnnData and OME Zarr files for vitessce
    adata = sdata[f"square_{bin_size_um:03}um"]

    #
    num_cells = adata.obs.shape[0]
    adata.obsm["segmentations"] = np.zeros((num_cells, 4, 2))
    for i in range(num_cells):
        adata.obsm["segmentations"][i, :, :] = _to_square(
            adata.obsm["spatial"][i, 0], adata.obsm["spatial"][i, 1], bin_side_length
        )

    #
    adata = optimize_adata(
        adata,
        #obs_cols=["location_id"],   # CHECKME: some other columns? total_counts?
        obsm_keys=[ "segmentations", "spatial" ],
        optimize_X=True,
        # Vitessce plays nicely with dense matrices saved with chunking
        # and this one is small enough that dense is not a huge overhead.
        to_dense_X=True,
    )

    # Write AnnData zarr. It will be used by Vitessce as source data.
    # (Ideally we would directly use the original SpatialData object from Harpy in Vitessce, but among other potential problems,
    # Harpy uses the latest spatialdata version, which writes zarr v3, and Vitessce currently (early Dec 2025) does not support this).
    adata.write_zarr(output_path_adata, chunks=[adata.shape[0], VAR_CHUNK_SIZE])

    # Extract full resolution H&E image to separate OME Zarr store
    array = sdata[f"{dataset_id}_full_image" ][ "scale0" ][ "image" ].data.compute()
    multiplex_img_to_ome_zarr(img_arr=array, channel_names=[ "r", "g", "b" ], axes="cyx",  output_path=output_path_img)

    # Extract tissue mask to separate OME Zarr store
    array=sdata[f"{dataset_id}_tissue_mask" ].data.compute()
    multiplex_img_to_ome_zarr(img_arr=array[None, ...], channel_names=["tissue"], axes="cyx",  output_path=output_path_tissue_mask)


def _make_vitessce_config(name: str,
                          description: str,
                          output_path_adata: str,
                          output_path_img: str,
                          output_path_tissue_mask: str) -> VitessceConfig:
    
    vc = VitessceConfig(schema_version="1.0.15",
                        name=name,
                        description=description)

    dataset = vc.add_dataset(name=__name__)

    dataset.add_object(
        MultiImageWrapper(
            image_wrappers=[
                OmeZarrWrapper(img_path=output_path_img, is_bitmask=False, name='H&E Image'),
                # OmeZarrWrapper(img_path=output_path_tissue_mask, is_bitmask=True, name='Tissue Mask'),
            ]
        ),
    )

    dataset.add_object(AnnDataWrapper(
            adata_path=output_path_adata,
            obs_locations_path="obsm/spatial",
            obs_segmentations_path="obsm/segmentations",
            obs_feature_matrix_path="X",
            coordination_values={
                "obsType": "spot",  # tell Vitessce to consider each observation to be a "spot"
                "featureType": "gene",
                "featureValueType": "expression"
            }
        ),
    )

    spatial = vc.add_view(cm.SPATIAL, dataset=dataset)
    obs_sets = vc.add_view(cm.OBS_SETS, dataset=dataset)
    layer_controller = vc.add_view(cm.LAYER_CONTROLLER, dataset=dataset)
    feature_list = vc.add_view(cm.FEATURE_LIST, dataset=dataset)
    description = vc.add_view(cm.DESCRIPTION, dataset=dataset)

    vc.link_views([spatial, layer_controller, feature_list, obs_sets], ['obsType'], ['spot'])
    vc.layout((spatial | (description / layer_controller / feature_list)))
    return vc


In [ ]:
datasets = [
    Dataset("SCA003", "SCA003: Visium HD FFPE, brain WT")
]

bin_size_um = 32


In [ ]:
root_dir = r"D:\2024-07-UTBenchmark-SpC\data"

for dataset in datasets:
    spatialdata_filepath = os.path.join(data_dir, "harpy", "sdata.zarr")
    data_dir = os.path.join(root_dir, "processed", "Visium_HD", dataset.name)
    spaceranger_output_path = os.path.join(root_dir, f"processed/Visium_HD/{dataset.name}/SpaceRanger")
    output_dir = os.path.join(data_dir, "vitessce")

    make_vitessce_dataset(dataset.name,
                          dataset.description,
                          spatialdata_filepath,
                          spaceranger_output_path,
                          bin_size_um,
                          output_dir)

In [ ]:
# IMPROVEME: can we hide the tissue mask by default?
# IMPROVEME: can we list the H&E image UI controls first, and below the ones for tissue?
# IMPROVEME: remove r/g/b channel sliders in the H&E image?
# IMPROVEME: show a predefined gene by default (maybe one of the mitochondrial genes that are expressed almost everywhere in the tissue, like mt-Atp6?)
# IMPROVEME: can we some only show the bins inside the tissue?
# IMPROVEME: make the H&E image panel wider than the default half of the screen
# IMPROVEME: can we avoid that vc.export(to="files") makes copies of zarr files that we already have somewhere else? 
#            I think so, see https://python-docs.vitessce.io/notebooks/widget_brain_with_base_dir.html